# sparse_release_bc_warmstart_screen_v1

Notebook-first BC warm-start screen for the sparse SOXX/SOXL release target revision. The previous notebook `sparse_rebound_release_target_revision_v1` restored effective rebound lift and passed fold-local target scope plus BC label-distribution readiness. This notebook checks the next gate before PPO: can a behavior-cloning actor actually fit the sparse release slice without needing extra weighting that may overfit?

Governance fields:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `sparse_rebound_release_target_revision_v1`.
- **Role**: `diagnose`.
- **Changed layer**: `model_selection`, `reward`, `feature`, `sampling`; no PPO and no runtime promotion here.
- **Target window**: fold-local train targets only; fixed-split and walk-forward evaluation remain future gates.
- **Accepted trade-off**: prefer the simplest BC weighting recipe that passes release-slice fit; avoid heavier weighting unless the sparse slice fails.
- **Failure mode**: global BC accuracy passes while release/rebound/downtrend slice fit fails; in that case do not hand off to PPO.

External references retained for this design:

- Qlib: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Focal Loss: https://arxiv.org/abs/1708.02002
- Prioritized Experience Replay: https://arxiv.org/abs/1511.05952
- Probability of Backtest Overfitting / CSCV: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    summarize_bc_slice_fit,
)
from src.soxl_quant.config import load_profile

EXPERIMENT = 'sparse_release_bc_warmstart_screen_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime/model_selection/card/sparse_rebound_release_target_revision_v1'
TARGETS_PATH = SOURCE_CARD / 'sparse_release_revised_targets.csv'
SOURCE_SELECTION_PATH = SOURCE_CARD / 'selection.json'
PROFILE_NAME = 'risk_budget_usage_v1'
FEATURE_FRAME_PROFILE = 'balanced_rule_residual_v22'
FRAME_PATH = ROOT / 'data' / f'soxl_profile_{FEATURE_FRAME_PROFILE}.pkl'
CREATED_AT = datetime.now(timezone.utc).isoformat()
CARD.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
RELEASE_SLICE_ACCURACY_MIN = 0.60
RELEASE_SLICE_SAMPLES_MIN = 20
GLOBAL_ACCURACY_MIN = 0.70


def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if pd.isna(value):
        return None
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def to_json_safe(value):
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [to_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(
        json.dumps(to_json_safe(payload), indent=2, sort_keys=True, default=json_default, allow_nan=False) + '\n',
        encoding='utf-8',
    )


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
source_selection = json.loads(SOURCE_SELECTION_PATH.read_text())
frame = pd.read_pickle(FRAME_PATH)
profile = load_profile(PROFILE_NAME)

coverage = {
    'experiment': EXPERIMENT,
    'source_targets': str(TARGETS_PATH.relative_to(ROOT)),
    'source_decision': source_selection.get('decision'),
    'source_bc_training_allowed_next': bool(source_selection.get('bc_training_allowed_next', False)),
    'profile': PROFILE_NAME,
    'feature_frame_profile': FEATURE_FRAME_PROFILE,
    'target_rows': int(len(targets)),
    'date_start': str(pd.to_datetime(targets['datetime']).min().date()),
    'date_end': str(pd.to_datetime(targets['datetime']).max().date()),
    'release_gate_rows': int(targets.get('release_gate_state', pd.Series(False, index=targets.index)).fillna(False).astype(bool).sum()),
    'folds': sorted(targets['fold'].dropna().astype(str).unique().tolist()),
}
write_json(CARD / 'coverage.json', coverage)
coverage


{'experiment': 'sparse_release_bc_warmstart_screen_v1',
 'source_targets': 'runtime/model_selection/card/sparse_rebound_release_target_revision_v1/sparse_release_revised_targets.csv',
 'source_decision': 'sparse_release_targets_bc_screen_ready_no_training',
 'source_bc_training_allowed_next': True,
 'profile': 'risk_budget_usage_v1',
 'feature_frame_profile': 'balanced_rule_residual_v22',
 'target_rows': 1261,
 'date_start': '2020-01-02',
 'date_end': '2022-12-30',
 'release_gate_rows': 80,
 'folds': ['wf_2022_train', 'wf_2023_2024_train']}

In [3]:
dataset = build_state_objective_bc_dataset_from_frame(
    frame,
    targets=targets,
    profile=profile,
    sample_size=0,
)

dataset_summary = {
    'samples': int(len(dataset.actions)),
    'obs_shape': [int(v) for v in dataset.obs.shape],
    'unique_actions': int(pd.Series(dataset.actions).nunique()) if len(dataset.actions) else 0,
    'release_gate_samples': int(dataset.metadata.get('release_gate_state', pd.Series(False, index=dataset.metadata.index)).fillna(False).astype(bool).sum()) if not dataset.metadata.empty else 0,
    'rebound_samples': int(dataset.metadata.get('is_rebound_cluster', pd.Series(False, index=dataset.metadata.index)).fillna(False).astype(bool).sum()) if not dataset.metadata.empty else 0,
    'downtrend_samples': int(dataset.metadata.get('is_downtrend_state', pd.Series(False, index=dataset.metadata.index)).fillna(False).astype(bool).sum()) if not dataset.metadata.empty else 0,
}
write_json(CARD / 'dataset_summary.json', dataset_summary)
dataset_summary


{'samples': 755,
 'obs_shape': [755, 315],
 'unique_actions': 5,
 'release_gate_samples': 40,
 'rebound_samples': 66,
 'downtrend_samples': 477}

In [4]:
train_config = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=42)
weight_configs = [
    {
        'config': 'plain',
        'sample_weight_config': None,
        'complexity_rank': 0,
    },
    {
        'config': 'rebound4',
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(rebound_multiplier=4.0, downtrend_multiplier=1.0),
        'complexity_rank': 1,
    },
    {
        'config': 'rebound8_cb99',
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=8.0,
            downtrend_multiplier=1.0,
            class_balanced_beta=0.99,
        ),
        'complexity_rank': 2,
    },
]

fit_rows = []
slice_frames = []
for cfg in weight_configs:
    result = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=targets,
        profile=profile,
        train_config=train_config,
        sample_weight_config=cfg['sample_weight_config'],
        sample_size=0,
    )
    slice_metrics = evaluate_bc_actor_slice_fit(
        result.actor,
        result.obs,
        result.actions,
        dataset.metadata,
        slice_cols=('release_gate_state', 'is_rebound_cluster', 'is_downtrend_state'),
    )
    slice_summary = summarize_bc_slice_fit(
        slice_metrics,
        required_slices=('release_gate_state',),
        min_accuracy=RELEASE_SLICE_ACCURACY_MIN,
        min_samples=RELEASE_SLICE_SAMPLES_MIN,
    )
    release_row = slice_metrics.set_index('slice').loc['release_gate_state'].to_dict()
    action_counts = pd.Series(result.actions).value_counts(normalize=True)
    row = {
        'config': cfg['config'],
        'complexity_rank': int(cfg['complexity_rank']),
        'slice_fit_gate': bool(slice_summary['slice_fit_gate']),
        'release_accuracy': float(release_row['accuracy']),
        'release_samples': int(release_row['samples']),
        'release_predicted_dominant_action': int(release_row['predicted_dominant_action']),
        'release_predicted_dominant_share': float(release_row['predicted_dominant_share']),
        'unique_actions': int((action_counts > 0).sum()),
        'dominant_action_share': float(action_counts.max()) if len(action_counts) else 1.0,
        **{k: v for k, v in result.metrics.items() if isinstance(v, (int, float, np.integer, np.floating))},
    }
    row['global_fit_gate'] = bool(row['after_accuracy'] >= GLOBAL_ACCURACY_MIN)
    row['candidate_gate'] = bool(row['global_fit_gate'] and row['slice_fit_gate'])
    fit_rows.append(row)
    tagged = slice_metrics.copy()
    tagged.insert(0, 'config', cfg['config'])
    slice_frames.append(tagged)

fit_screen = pd.DataFrame(fit_rows).sort_values(
    ['candidate_gate', 'complexity_rank', 'after_accuracy', 'release_accuracy'],
    ascending=[False, True, False, False],
).reset_index(drop=True)
slice_metrics = pd.concat(slice_frames, ignore_index=True)
fit_screen.to_csv(CARD / 'bc_weight_screen.csv', index=False)
slice_metrics.to_csv(CARD / 'state_slice_bc_metrics.csv', index=False)

display(fit_screen)
display(slice_metrics)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


,config,complexity_rank,slice_fit_gate,release_accuracy,release_samples,release_predicted_dominant_action,release_predicted_dominant_share,unique_actions,dominant_action_share,samples,...,before_accuracy,train_last_loss,train_last_batch_accuracy,after_loss,after_accuracy,global_fit_gate,candidate_gate,weight_mean,weight_min,weight_max
0,plain,0,True,0.975,40,4,0.975,5,0.295364,755,...,0.205298,1.239666,0.679688,1.155358,0.765563,True,True,NaN,NaN,NaN
1,rebound4,1,True,0.975,40,4,0.975,5,0.295364,755,...,0.205298,1.232830,0.617188,1.189867,0.724503,True,True,1.0,0.792235,3.168940
2,rebound8_cb99,2,True,1.000,40,4,1.000,5,0.295364,755,...,0.205298,1.188085,0.656250,1.172519,0.733775,True,True,1.0,0.542921,4.930356


,config,slice,samples,accuracy,loss,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share
0,plain,all,755,0.765563,1.155358,5,4,1,0.295364,2,0.336424
1,plain,is_downtrend_state,477,0.798742,1.120358,5,4,1,0.383648,1,0.400419
2,plain,is_rebound_cluster,66,0.924242,0.987778,2,2,4,0.803030,4,0.848485
3,plain,release_gate_state,40,0.975000,0.935883,1,2,4,1.000000,4,0.975000
4,rebound4,all,755,0.724503,1.189867,5,4,1,0.295364,2,0.398675
5,rebound4,is_downtrend_state,477,0.750524,1.167120,5,4,1,0.383648,2,0.433962
6,rebound4,is_rebound_cluster,66,0.924242,0.982023,2,2,4,0.803030,4,0.848485
7,rebound4,release_gate_state,40,0.975000,0.930659,1,2,4,1.000000,4,0.975000
8,rebound8_cb99,all,755,0.733775,1.172519,5,5,1,0.295364,2,0.350993
9,rebound8_cb99,is_downtrend_state,477,0.748428,1.160751,5,5,1,0.383648,2,0.417191


In [5]:
best = fit_screen.iloc[0].to_dict() if not fit_screen.empty else {}
best_config = str(best.get('config', ''))
best_slice_metrics = slice_metrics[slice_metrics['config'].astype(str) == best_config].copy()
best_slice_summary = summarize_bc_slice_fit(
    best_slice_metrics.drop(columns=['config']),
    required_slices=('release_gate_state',),
    min_accuracy=RELEASE_SLICE_ACCURACY_MIN,
    min_samples=RELEASE_SLICE_SAMPLES_MIN,
)
write_json(CARD / 'bc_slice_summary.json', best_slice_summary)

if bool(best.get('candidate_gate', False)):
    decision = 'sparse_release_bc_warmstart_ready_for_ppo_screen'
    next_mechanism = 'run_fold_local_ppo_from_plain_sparse_release_bc_then_fixed_split_walkforward'
elif bool(best.get('global_fit_gate', False)):
    decision = 'sparse_release_bc_global_fit_slice_failed_no_ppo'
    next_mechanism = 'retune_release_slice_weighting_or_auxiliary_state_head'
else:
    decision = 'sparse_release_bc_fit_failed_no_ppo'
    next_mechanism = 'revise_target_mapping_or_training_surface_before_ppo'

selection = {
    'experiment': EXPERIMENT,
    'created_at': CREATED_AT,
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'sparse_rebound_release_target_revision_v1',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'reward', 'feature', 'sampling'],
    'source_targets': str(TARGETS_PATH.relative_to(ROOT)),
    'profile': PROFILE_NAME,
    'feature_frame_profile': FEATURE_FRAME_PROFILE,
    'training_allowed_next': bool(best.get('candidate_gate', False)),
    'promotion_allowed': False,
    'promotion_blocker': 'BC slice fit is not a fixed-split or walk-forward backtest; PPO and +30pp / <=15% MaxDD gates remain required.',
    'best_config': best_config,
    'best_after_accuracy': float(best.get('after_accuracy', np.nan)),
    'best_before_accuracy': float(best.get('before_accuracy', np.nan)),
    'best_after_loss': float(best.get('after_loss', np.nan)),
    'best_release_accuracy': float(best.get('release_accuracy', np.nan)),
    'best_release_samples': int(best.get('release_samples', 0)),
    'best_slice_fit_gate': bool(best.get('slice_fit_gate', False)),
    'best_global_fit_gate': bool(best.get('global_fit_gate', False)),
    'best_unique_actions': int(best.get('unique_actions', 0)),
    'best_dominant_action_share': float(best.get('dominant_action_share', 1.0)),
    'selected_policy': 'plain_bc' if best_config == 'plain' else 'weighted_bc',
    'overfit_control': 'prefer_lowest_complexity_candidate_that_passes_release_slice_gate',
    'acceptance_gate_for_next_run': {
        'release_slice_accuracy_min': RELEASE_SLICE_ACCURACY_MIN,
        'release_slice_samples_min': RELEASE_SLICE_SAMPLES_MIN,
        'global_accuracy_min': GLOBAL_ACCURACY_MIN,
        'fixed_split_required_after_bc': True,
        'walkforward_required_after_bc': True,
        'relative_return_min_pp': TARGET_RELATIVE_RETURN,
        'max_dd_max_pct': TARGET_MAX_DD,
        'no_runtime_promotion_from_bc_only': True,
    },
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://arxiv.org/abs/1708.02002',
        'https://arxiv.org/abs/1511.05952',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
    'sidecars': [
        'coverage.json',
        'dataset_summary.json',
        'bc_weight_screen.csv',
        'state_slice_bc_metrics.csv',
        'bc_slice_summary.json',
        'selection.json',
        'summary.json',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'training_allowed_next': selection['training_allowed_next'],
    'conclusion': [
        f"Best BC config is {best_config}; it uses the lowest-complexity recipe that passes the release-slice gate.",
        f"Global BC accuracy improved from {selection['best_before_accuracy']:.3f} to {selection['best_after_accuracy']:.3f}; release-slice accuracy is {selection['best_release_accuracy']:.3f} on {selection['best_release_samples']} samples.",
        f"Dominant label share is {selection['best_dominant_action_share']:.3f} with {selection['best_unique_actions']} actions, so the action surface is not degenerate.",
        'Conclusion: plain sparse-release BC is ready for a PPO screen, but there is still no promotion without fixed-split and walk-forward +30pp / <=15% MaxDD validation.',
    ],
    'coverage': coverage,
    'dataset_summary': dataset_summary,
    'best': best,
    'slice_summary': best_slice_summary,
    'selection': selection,
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
for line in summary['conclusion']:
    display(Markdown(f'- {line}'))
selection


- Best BC config is plain; it uses the lowest-complexity recipe that passes the release-slice gate.

- Global BC accuracy improved from 0.205 to 0.766; release-slice accuracy is 0.975 on 40 samples.

- Dominant label share is 0.295 with 5 actions, so the action surface is not degenerate.

- Conclusion: plain sparse-release BC is ready for a PPO screen, but there is still no promotion without fixed-split and walk-forward +30pp / <=15% MaxDD validation.

{'experiment': 'sparse_release_bc_warmstart_screen_v1',
 'created_at': '2026-05-25T18:18:09.361599+00:00',
 'decision': 'sparse_release_bc_warmstart_ready_for_ppo_screen',
 'next_mechanism': 'run_fold_local_ppo_from_plain_sparse_release_bc_then_fixed_split_walkforward',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'sparse_rebound_release_target_revision_v1',
 'role': 'diagnose',
 'changed_layer': ['model_selection', 'reward', 'feature', 'sampling'],
 'source_targets': 'runtime/model_selection/card/sparse_rebound_release_target_revision_v1/sparse_release_revised_targets.csv',
 'profile': 'risk_budget_usage_v1',
 'feature_frame_profile': 'balanced_rule_residual_v22',
 'training_allowed_next': True,
 'promotion_allowed': False,
 'promotion_blocker': 'BC slice fit is not a fixed-split or walk-forward backtest; PPO and +30pp / <=15% MaxDD gates remain required.',
 'best_config': 'plain',
 'best_after_accuracy': 0.7655629139072848,
 'best_bef